# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'external product/partner',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
select_relevant_links("https://huggingface.com")

{'links': [{'type': 'about page', 'url': 'https://huggingface.com/brand'},
  {'type': 'company page', 'url': 'https://huggingface.com/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.com/blog'},
  {'type': 'partnerships page',
   'url': 'https://huggingface.com/organizations'},
  {'type': 'pricing page', 'url': 'https://huggingface.com/pricing'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co/'}]}

In [10]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub repository', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Community / forum', 'url': 'https://discuss.huggingface.co/'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
convaiinnovations/laya
Updated
6 days ago
•
4.48k
Edge0/Audio8-ASR-Infinite
Updated
6 days ago
•
23.7k
•
1.48k
abenzerps/Qwen-Image-2.1-Uncensored-GGUF
Updated
1 day ago
•
1.15M
•
2.39k
Qwen/Qwen-Image-2.1
Updated
9 days ago
•
64.4k
•
2.64k
XingChen-AGI/TeleOCR
Updated
about 18 hours ago
•
30.4k
•
861
Browse 2M+ models
Spaces
Running
Featured
319
Jev Decision Index
🔬
319
Benchmark

In [18]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information. Keep this brochure short and concise.
"""


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nconvaiinnovations/laya\nUpdated\n6 days ago\n•\n4.48k\nEdge0/Audio8-ASR-Infinite\nUpdated\n6 days ago\n•\n23.7k\n•\n

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Hugging the Future (Literally) 🤗

---

## Welcome to Hugging Face!

Where else can you find **2 million+ models**, **500k+ datasets**, and a buzzing community all in one place? Hugging Face is *the* AI playground — a collaboration hub where the brightest minds in machine learning (ML) gather to share, build, and innovate together. If AI was a party, Hugging Face would be the dance floor—electric, expansive, and inclusive.

---

## What’s the Big Deal?

- **Models Galore:** Need an image generator? A chatbot? A speech recognition engine? With over 2 million models, Hugging Face has a little something for every AI appetite.
  
- **Datasets to Die For:** Explore half a million datasets ranging from doctor-patient conversations to the entire Wikipedia — because your AI deserves the best fuel.

- **Spaces & Apps:** Run and discover cutting-edge AI applications on the fly. Want video with synchronized soundtrack or image editing at the speed of thought? There’s a space for that.

- **Infinite Collaboration:** Host unlimited projects, fork repos, run experiments, and join a community that’s building the future, *together.*

---

## Who’s Hugging?

- **Developers & Researchers:** Dive deep, share your models, or find the perfect dataset for your next breakthrough.  
- **Enterprises:** Get tailor-made support, inference endpoints, and enterprise-grade tools to power your AI solutions.  
- **AI Enthusiasts & Learners:** Tap into docs, daily papers, and lively forums to learn and grow your AI chops.

---

## Culture & Careers: Join the Hug!

At Hugging Face, we’re more than an AI platform — we’re a **community with a cause**. Collaboration and openness aren’t buzzwords here; they’re the air we breathe. Expect to work where curiosity drives innovation, diverse voices power ideas, and everyone gets a virtual *hug* (okay, maybe just a warm welcome).

- **Careers:** If you love AI, open source, and pushing boundaries, you’ll fit right in. From software engineers to community builders, Hugging Face is hiring the future.

- **Community Spirit:** Join us on Discord, GitHub, or forums. Contribute code, share datasets, or just geek out over the latest AI paper — all with a supportive, can-do vibe.

---

## Why Hugging Face?

- Because AI isn’t magic; it’s *collaborative magic*.  
- Because we believe in open, accessible, and responsible AI for everyone.  
- Because nerds with heart build the best tech.  
- And honestly? Because hugging is way better than just shaking hands.

---

### Ready to jump in?

- **Explore AI apps:** Tap into the growing arsenal of tools powering the AI revolution.  
- **Browse 2M+ models:** Because AI is better when shared.  
- **Join the community:** Whether you’re a newbie or a seasoned pro, everyone’s welcome.

---

**Hugging Face** — The AI community where innovation meets open arms.  
*(No robots were harmed in making this brochure)*

---

Follow us: [Discord] [GitHub] [Forum] | Sign Up now and get your first Hug! 🤗

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Playground Where Models Party and Innovation Never Sleeps!

---

## Who Are We?  
Welcome to **Hugging Face**, the cozy AI community shaking up the future one model at a time. Think of us as the bustling digital café for machine learning enthusiasts, researchers, and developers - where fuzzy ideas turn into sharp, cutting-edge AI tools. Whether you’re a coder, data scientist, or just AI-curious, we’ve got a place at the table for you.

---

## What We Do:  
- **2 Million+ Models & Counting!**  
  Why settle for just Google when you can browse our massive smorgasbord of AI models? From cutting-edge conversational agents to image generators, we host it all.

- **Datasets Galore**  
  With over *half a million* curated datasets, if data is the new oil, think of us as your AI gas station—full tank guaranteed.

- **Spaces**  
  Your playground for AI apps — from video generation synched with soundtracks to image editing magic. Try before you build!

- **HuggingChat & More**  
  Experiment with live AI chat models, test rockstar algorithms, and collaborate like you mean it. 

---

## Who Loves Us?  
- **Researchers & Developers** love our easy-to-navigate model zoo and seamless collaboration tools.  
- **Enterprises** appreciate our robust PRO support, storage buckets, and scalable inference endpoints.  
- **Students & Educators** find a treasure trove of datasets and learning resources.  
- **AI Hobbyists** get inspired daily, hanging out on Discord or diving into the latest blogs and papers.

---

## Culture: The Heart of Hugging Face  
- **Community-First**: Whether it’s GitHub discussions or our lively forum, we believe AI will thrive best when we build it *together*.  
- **Open & Inclusive**: Hands down, no secret handshakes here — just plenty of shared code, ideas, and GIFs.  
- **Innovation Playground**: Experiment, fail fast, learn faster, and share successes.  
- **Fun meets Serious**: Because who said creating the future can’t involve a few laughs?

---

## Why Work With Us?  
- Cutting-edge tech stack with **influence on the global AI stage**.  
- Collaborative environment where your ideas are as welcome as your favorite coffee blend.  
- Opportunities to scale your career while scaling models used by millions.  
- Be part of a mission to democratize AI — powering tools that anyone, anywhere, can use.

---

## Pricing & Enterprise Goodies  
Our pricing is as flexible as our models! From *free community access* to tailored **Team & Enterprise Plans**, we scale with your ambitions. Enterprise customers enjoy perks like dedicated support, private buckets to secure your precious data, and fully managed inference endpoints for hassle-free AI deployment.

---

## Join The AI Revolution—With a Hug!  
- Visit: [huggingface.co](https://huggingface.co)  
- Join the Party: Chat on Discord, contribute on GitHub, learn or teach.  
- Whether you're a wizard coder, curious beginner, or business visionary — your AI future starts here.

---

Hugging Face — *where the models are smart, and the community is heart*.  
Ready to get your hands on the future? Let's make some magic! ✨🤗🤖

---

# Your AI Adventure Awaits!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>